In [ ]:
import numpy as np                     # Untuk operasi numerik dan array
import matplotlib.pyplot as plt        # Untuk visualisasi data
import tensorflow as tf                # Framework deep learning
from tensorflow.keras import layers, models  # API untuk membangun arsitektur model
from tensorflow.keras.datasets import mnist  # Dataset digit tulisan tangan
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping  # Callback untuk optimasi training

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalisasi data (0-1)
x_train = x_train.reshape((-1, 28, 28, 1))  # Reshape untuk format CNN
x_test = x_test.reshape((-1, 28, 28, 1))

# Visualization of random training data
plt.figure(figsize=(10, 2))
random_indices = np.random.randint(0, len(x_train), 5)
for i, idx in enumerate(random_indices):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_train[idx].reshape(28, 28), cmap='gray')
    plt.title(f"{y_train[idx]}")
    plt.axis('off')
plt.show()

model = models.Sequential([
    # Layer 1: Konvolusi + Normalisasi + Pooling
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 2: Konvolusi + Normalisasi + Pooling
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 3: Konvolusi + Normalisasi
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    
    # Fully Connected Layers
    layers.Flatten(),
    layers.Dropout(0.25),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(10, activation='softmax')  # Output layer (10 digit)
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint(  # Menyimpan model terbaik
    'best_model.weights.h5',  
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    save_weights_only=True,
    verbose=1
)

early_stopping = EarlyStopping(  # Menghentikan training jika tidak ada perbaikan
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True,
    mode='max'
)

history = model.fit(
    x_train, y_train, 
    epochs=5, 
    validation_data=(x_test, y_test),
    callbacks=[checkpoint, early_stopping]
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

# Visualize model predictions on test samples
predictions = model.predict(x_test[:10], verbose=0)
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    pred = np.argmax(predictions[i])
    true_val = y_test[i]
    title_color = 'green' if pred == true_val else 'red'
    plt.title(f"Pred: {pred}\nTrue: {true_val}", color=title_color)
    plt.axis('off')
plt.tight_layout()
plt.show()

: 